## Base v24 — Autores BR domínio público via Wikisource pt

Download automático (API do pt.wikisource) das obras de autores brasileiros em domínio público, sem retreinar tokenizer.
Versionamento idêntico ao playground_3: **nunca sobrescreve**, gera vN+1 a partir da última base (`df_full_v*.pq`).
Autores: Machado de Assis, Aluísio Azevedo, Álvares de Azevedo, João do Rio, Coelho Neto, Augusto dos Anjos,
Afonso Arinos, Raul Pompeia, Rodolfo Teófilo, Monteiro Lobato, Olavo Bilac, Bernardo Guimarães, Cruz e Sousa, Humberto de Campos.

In [1]:
# CONFIG
AUTHORS = {
    'machado':      'Machado de Assis',
    'cruz_sousa':   'Cruz e Sousa',
    'bilac':        'Olavo Bilac',
    'campos':       'Humberto de Campos',
    'augusto_anjos':'Augusto dos Anjos',
    'alvares':      'Álvares de Azevedo',
    'lobato':       'Monteiro Lobato',
    'guimaraes':    'Bernardo Guimarães',
    'aluisio':      'Aluísio Azevedo',
    'joao_rio':     'João do Rio',
    'coelho_neto':  'Coelho Neto',
    'pompeia':      'Raul Pompeia',
    'arinos':       'Afonso Arinos',
    'teofilo':      None,  # sem categoria; páginas via busca
}
AUTHORS_TITLES = {'teofilo': ['O Reino de Kiato']}  # capítulos via busca
TOKENIZER = "artifacts/tokenizers/gpt2_ptbr_50k_v2"
WEIGHT_CLIP = 500_000
SPLIT_FRAC = 0.85
RANDOM_STATE = 1
SKIP_DOWNLOAD = True  # False = baixar do Wikisource de novo (lento, 429s); True = usar cache data/ws_*
import os
from pathlib import Path
for k in AUTHORS:
    Path(f'data/ws_{k}').mkdir(parents=True, exist_ok=True)
print('autores:', len(AUTHORS), '| SKIP_DOWNLOAD:', SKIP_DOWNLOAD)

autores: 14 | SKIP_DOWNLOAD: True


In [2]:
# Versionamento automático (igual playground_3)
import glob, re
vers = sorted(int(re.search(r'df_full_v(\d+)\.pq', p).group(1)) for p in glob.glob('data/df_full_v*.pq'))
BASE_TXT = f'data/df_full_v{max(vers)}.pq'
BASE_ENC = f'data/df_full_encoded_v{max(vers)}.pq'
NEXT_VERSION = max(vers) + 1
OUT_TXT = f'data/df_full_v{NEXT_VERSION}.pq'
OUT_ENC = f'data/df_full_encoded_v{NEXT_VERSION}.pq'
print(f'base atual: {BASE_TXT} -> gera v{NEXT_VERSION}')

base atual: data/df_full_v23.pq -> gera v24


In [3]:
# Downloader do Wikisource pt (SKIP_DOWNLOAD=True -> só enumera o cache, sem tocar na API)
import urllib.request, urllib.parse, json, time, re
from bs4 import BeautifulSoup

API = 'https://pt.wikisource.org/w/api.php'

def api_get(params, retries=6):
    url = API + '?' + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (base-prep)'})
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=45) as r:
                return json.load(r)
        except urllib.error.HTTPError as e:
            if e.code == 429 or e.code >= 500:
                wait = 4 * (2 ** attempt)
                print(f'  [rate-limit {e.code}] esperando {wait}s...')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f'rate-limit persistente em {url}')

def cat_members(cat):
    pages, cont = [], {}
    while True:
        d = api_get({'action':'query','list':'categorymembers','cmtitle':'Categoria:'+cat,
                     'cmlimit':'500','format':'json', **cont})
        pages += [m['title'] for m in d.get('query',{}).get('categorymembers',[]) if m['ns']==0]
        if 'continue' not in d: break
        cont = {'cmcontinue': d['continue']['cmcontinue']}
        time.sleep(0.3)
    return sorted(set(pages))

def fetch_text(title):
    d = api_get({'action':'parse','page':title,'prop':'text','format':'json','redirects':'1'})
    soup = BeautifulSoup(d['parse']['text']['*'], 'html.parser')
    main = soup.select_one('#mw-content-text') or soup
    for tag in main.select('.editsection, .mw-editsection, .noprint, .printfooter, .catlinks'):
        tag.decompose()
    return main.get_text('\n')

def clean_title(t):
    return re.sub(r'[\\/:*?"<>|]', '_', t)

if SKIP_DOWNLOAD:
    total_cache = 0
    for key in AUTHORS:
        n = len(list(Path(f'data/ws_{key}').glob('*.txt')))
        total_cache += n
        print(f'{key}: {n} arquivos em cache')
    print(f'TOTAL em cache: {total_cache} (modo SKIP_DOWNLOAD — nada baixado)')
else:
    downloaded = 0
    for key, cat in AUTHORS.items():
        if cat:
            pages = cat_members(cat)
        else:
            d = api_get({'action':'query','list':'search','srsearch':AUTHORS_TITLES[key][0],'srnamespace':'0','srlimit':'50','format':'json'})
            pages = [r['title'] for r in d.get('query',{}).get('search',[])] + AUTHORS_TITLES[key]
        pages = [p for p in pages if '/' not in p]  # top-level (página-mãe transpila capítulos)
        n_ok = 0
        for p in pages:
            fp = f'data/ws_{key}/{clean_title(p)}.txt'
            if os.path.exists(fp):
                continue
            try:
                txt = fetch_text(p)
                if len(txt.strip()) < 800:
                    continue
                open(fp, 'w', encoding='utf-8').write(txt)
                n_ok += 1
                time.sleep(0.35)
            except Exception as e:
                print(f'  ERRO {key} | {p}: {e}')
        print(f'{key}: {len(pages)} obras -> {n_ok} baixadas')
        downloaded += n_ok
    print(f'\nTOTAL: {downloaded} páginas novas baixadas')

machado: 349 arquivos em cache
cruz_sousa: 166 arquivos em cache
bilac: 99 arquivos em cache
campos: 171 arquivos em cache
augusto_anjos: 15 arquivos em cache
alvares: 90 arquivos em cache
lobato: 20 arquivos em cache
guimaraes: 11 arquivos em cache
aluisio: 9 arquivos em cache
joao_rio: 11 arquivos em cache
coelho_neto: 4 arquivos em cache
pompeia: 4 arquivos em cache
arinos: 2 arquivos em cache
teofilo: 1 arquivos em cache
TOTAL em cache: 952 (modo SKIP_DOWNLOAD — nada baixado)


In [4]:
# DataFrame dos novos autores
import pandas as pd
from src.prep import clean_text2

rows = []
for key in AUTHORS:
    for fp in sorted(Path(f'data/ws_{key}').glob('*.txt')):
        rows.append({'title': fp.stem, 'author': key, 'class': key, 'extension': 'ws',
                     'path_raw': f'https://pt.wikisource.org/wiki/{urllib.parse.quote(fp.stem)}',
                     'path_txt': str(fp), 'text': fp.read_text(encoding='utf-8')})
df_new = pd.DataFrame(rows)
df_new['subtitle'] = ''
df_new['text_len'] = df_new.text.str.len()
df_new['text_clean'] = df_new.text.apply(clean_text2)
df_new['text_clean_len'] = df_new.text_clean.str.len()
df_new = df_new[df_new.text_clean_len >= 500].reset_index(drop=True)
print('docs novos:', len(df_new), '| chars limpos M:', round(df_new.text_clean_len.sum()/1e6, 2))
print(df_new.groupby('author').agg(docs=('title','count'), charsM=('text_clean_len', lambda s: round(s.sum()/1e6, 2))))

docs novos: 952 | chars limpos M: 5.76
               docs  charsM
author                     
aluisio           9    0.02
alvares          90    0.37
arinos            2    0.02
augusto_anjos    15    0.03
bilac            99    0.20
campos          171    0.47
coelho_neto       4    0.00
cruz_sousa      166    0.41
guimaraes        11    0.02
joao_rio         11    0.08
lobato           20    0.22
machado         349    3.89
pompeia           4    0.03
teofilo           1    0.00


In [5]:
# Diagnóstico: ortografia arcaica (pré-reforma) e variante BR/PT-PT por autor
ARCAIC = ['ph', 'th', 'ch', 'y', 'annos', 'diccionario', 'escriptor', 'secção', 'licção', 'dous']
def arch_ratio(t):
    tl = t.lower()
    hits = sum(tl.count(m) for m in ARCAIC)
    return hits / max(len(tl), 1) * 1000  # por milhar de chars
PT_MARKS = ['facto,', 'direcção', 'connosco', 'comboio', 'telemóvel']
BR_MARKS = ['você', 'fato,', 'direção', 'celular']
def var(t):
    tl = t.lower()
    s = sum(tl.count(m) for m in BR_MARKS) - sum(tl.count(m) for m in PT_MARKS)
    return 'BR' if s >= 0 else 'PT-PT?'
df_new['arch_per_mil'] = df_new.text_clean.apply(arch_ratio)
df_new['variante'] = df_new.text_clean.apply(var)
print(df_new.groupby('author').agg(arquaico_por_mil=('arch_per_mil','mean'), variante=('variante', lambda s: s.mode().iloc[0])))

               arquaico_por_mil variante
author                                  
aluisio                0.543857       BR
alvares                1.832215       BR
arinos                 1.862042       BR
augusto_anjos          1.590021       BR
bilac                  1.751665       BR
campos                 1.907409       BR
coelho_neto            2.394990       BR
cruz_sousa             1.925071       BR
guimaraes              1.263805       BR
joao_rio               3.010707       BR
lobato                 2.664907       BR
machado                3.061352       BR
pompeia                3.850752       BR
teofilo                0.000000       BR


In [6]:
# Concat com a base atual (guarda: pula autores já presentes)
cols = ['title','author','extension','class','subtitle','path_raw','path_txt',
        'text_raw_len','text','text_len','text_clean','text_clean_len','weights','split']
df_base = pd.read_parquet(BASE_TXT)
dup = df_new[df_new.title.isin(set(df_base.title))]
if len(dup):
    print('duplicados por título (pulando):', len(dup))
    df_new = df_new[~df_new.title.isin(set(df_base.title))].copy()
df_new['weights'] = 0.0
df_new['split'] = ''
df_new['text_raw_len'] = df_new['text'].str.len()
df_full = pd.concat([df_base[cols], df_new[cols]], ignore_index=True)
df_full['text_len'] = df_full.text.str.len()
df_full['text_clean'] = df_full.text.apply(clean_text2)
df_full['text_clean_len'] = df_full.text_clean.str.len()
df_full['weights'] = df_full.text_clean_len.clip(0, WEIGHT_CLIP)
df_full['weights'] = df_full.weights / df_full.weights.sum()
idx_eval = df_full.sample(frac=1-SPLIT_FRAC, random_state=RANDOM_STATE, weights='weights').index
df_full['split'] = pd.Series(df_full.index.isin(idx_eval)).map({False:'train', True:'eval'})
df_full = df_full.sample(frac=1).reset_index(drop=True)
print('total docs:', len(df_full), '| chars limpos M:', round(df_full.text_clean_len.sum()/1e6, 2))

duplicados por título (pulando): 2


total docs: 1145 | chars limpos M: 77.22


In [7]:
# Salvar vN (texto) + encode com tokenizer v2
df_full.to_parquet(OUT_TXT)
import os
print('salvo:', OUT_TXT, round(os.path.getsize(OUT_TXT)/1e6, 1), 'MB')
from transformers import AutoTokenizer
from tqdm import tqdm
tqdm.pandas()
tok = AutoTokenizer.from_pretrained(TOKENIZER)
encode = lambda s: tok(s, truncation=False).input_ids
df_full['text_encoded'] = df_full.text_clean.progress_apply(encode)
df_full['text_encoded_len'] = df_full.text_encoded.apply(len)
df_full.to_parquet(OUT_ENC)
print('salvo:', OUT_ENC, round(os.path.getsize(OUT_ENC)/1e6, 1), 'MB')
print(f'tokens M: {df_full.text_encoded_len.sum()/1e6:.2f}')

salvo: data/df_full_v24.pq 99.7 MB


C:\Users\Bruno\.conda\envs\transformers-fun\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


  0%|          | 0/1145 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1448 > 1024). Running this sequence through the model will result in indexing errors


  3%|▎         | 33/1145 [00:00<00:14, 76.29it/s]

...

 95%|█████████▌| 1091/1145 [00:27<00:00, 105.60it/s]

 98%|█████████▊| 1119/1145 [00:27<00:00, 120.71it/s]

 99%|█████████▉| 1136/1145 [00:28<00:00, 73.58it/s] 

100%|██████████| 1145/1145 [00:28<00:00, 40.37it/s]

salvo: data/df_full_encoded_v24.pq 131.2 MB
tokens M: 17.42


In [8]:
# VALIDAÇÃO + BALANCEAMENTO (chars e tokens por autor)
df_full = pd.read_parquet(OUT_ENC)
g = df_full.groupby('author').agg(docs=('title','count'), chars=('text_clean_len','sum'), toks=('text_encoded_len','sum'))
g['% chars'] = (100*g.chars/g.chars.sum()).round(1)
g['% toks'] = (100*g.toks/g.toks.sum()).round(1)
g.chars = (g.chars/1e6).round(2); g.toks = (g.toks/1e6).round(2)
print(g.sort_values('toks', ascending=False).to_string())
df_old = pd.read_parquet(BASE_ENC)
print(f'\n{os.path.basename(BASE_ENC)}: {len(df_old)} docs, {df_old.text_encoded_len.sum()/1e6:.2f}M toks')
print(f'{os.path.basename(OUT_ENC)}: {len(df_full)} docs, {df_full.text_encoded_len.sum()/1e6:.2f}M toks')
print('PRÓXIMO PASSO: apontar yaml para', OUT_ENC)

               docs  chars   toks  % chars  % toks
author                                            
king             65  59.93  13.38     77.6    76.8
tolkien          10   4.96   1.09      6.4     6.2
machado         349   3.89   1.00      5.0     5.7
lovecraft       114   3.29   0.66      4.3     3.8
poe               5   2.80   0.65      3.6     3.7
campos          171   0.47   0.12      0.6     0.7
chambers          1   0.49   0.12      0.6     0.7
alvares          90   0.37   0.11      0.5     0.6
cruz_sousa      166   0.41   0.11      0.5     0.7
bilac            98   0.20   0.06      0.3     0.3
lobato           20   0.22   0.06      0.3     0.4
joao_rio         11   0.08   0.02      0.1     0.1
aluisio           9   0.02   0.01      0.0     0.0
arinos            2   0.02   0.01      0.0     0.0
augusto_anjos    14   0.03   0.01      0.0     0.1
pompeia           4   0.03   0.01      0.0     0.0
guimaraes        11   0.02   0.01      0.0     0.0
coelho_neto       4   0.00   0.


df_full_encoded_v23.pq: 195 docs, 15.91M toks
df_full_encoded_v24.pq: 1145 docs, 17.42M toks
PRÓXIMO PASSO: apontar yaml para data/df_full_encoded_v24.pq


### Observações
- **Ortografia arcaica**: obras do séc. XIX/começo do XX mantêm a grafia original no Wikisource (ex.: *Chrysalidas*, *Diccionario*). O diagnóstico por autor mostra o nível. Se algum autor poluir demais, dá pra excluir rodando de novo (o cache de download evita refazer a API).
- **Cruz e Sousa / Bilac / Campos**: poesia e crônicas — volume, fora do nicho horror (decisão do usuário).
- Histórico: v21 Tolkien → v22 Poe → v23 Chambers → **v24 autores BR Wikisource**.